# Relation Extraction from MediaWiki XML: Baseline + BiLSTM

This notebook implements two models for the character relationship Knowledge Graph project:

1. **TF-IDF features + Logistic Regression** as the baseline model.
2. **BiLSTM neural relation classifier** as the second model.

Pipeline

1. Set the XML input path, shared `data/` paths, model-specific output paths, relationship-label list, and optional LLM Judge settings.
2. Parse the MediaWiki XML and extract each character page using the page `<title>` as the character name and the revision `<text>` as the character text.
3. Clean wiki markup.
4. Build a character gazetteer from page titles.
5. Generate candidate character pairs.
6. Create an initial shared candidate file at `data/candidate_examples.csv` with weak labels from infobox fields and relationship keywords.
7. Optionally run a cached LLM Judge labeling step over `data/candidate_examples.csv` to replace weak labels with Gemma-generated labels.
8. Train and evaluate TF-IDF + Logistic Regression baseline variations.
9. Train and evaluate a BiLSTM classifier using entity-marked text.
10. Predict relationships for all candidates.
11. Aggregate predictions into Knowledge Graph edges.
12. Save shared preprocessing/training artifacts under `data/` and model-specific outputs under their model folders.
13. Create interactive PyVis graph visualizations.
14. Compare baseline and BiLSTM results.

By default, the notebook uses weak labels so the full pipeline can run cheaply. If `USE_LLM_JUDGE = True`, Section 8 creates or reuses `data/candidate_examples_llm_labeled.csv` and the downstream train/dev/test split uses those labels instead. LLM-generated labels are still not a gold human-annotated evaluation set, but they can be a stronger alternative to simple rule-based weak labels.


## 1. Configuration

In [ ]:
from pathlib import Path

# -----------------------------------------------------------------------------
# Shared data folder.
# -----------------------------------------------------------------------------
# All files that are inputs to, outputs from, or shared between multiple models
# are stored in data/. Model-specific artifacts remain in their own folders.
DATA_DIR = Path("data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# Input XML.
XML_PATH = DATA_DIR / "bookworm_09062026.xml"

# Shared preprocessing and training files used by both models.
PAGES_CSV = DATA_DIR / "pages.csv"
CHARACTER_GAZETTEER_CSV = DATA_DIR / "character_gazetteer.csv"
CANDIDATE_EXAMPLES_CSV = DATA_DIR / "candidate_examples.csv"
WEAK_LABEL_DISTRIBUTION_CSV = DATA_DIR / "label_distribution.csv"
TRAINING_LABEL_DISTRIBUTION_CSV = DATA_DIR / "training_label_distribution.csv"
TRAIN_CSV = DATA_DIR / "train.csv"
DEV_CSV = DATA_DIR / "dev.csv"
TEST_CSV = DATA_DIR / "test.csv"
SPLIT_LABEL_DISTRIBUTION_CSV = DATA_DIR / "split_label_distribution.csv"
MODEL_COMPARISON_CSV = DATA_DIR / "metrics_model_comparison.csv"
KG_MODEL_COMPARISON_CSV = DATA_DIR / "kg_model_comparison.csv"
RUN_SUMMARY_JSON = DATA_DIR / "model_run_summary.json"

# Model-specific output folders.
BASELINE_DIR = Path("baseline")  # TF-IDF + Logistic Regression artifacts.
BILSTM_DIR = Path("bilstm")      # BiLSTM artifacts.
BASELINE_DIR.mkdir(parents=True, exist_ok=True)
BILSTM_DIR.mkdir(parents=True, exist_ok=True)

# Relationship labels for this baseline. "no_relation" is needed as the negative class.
RELATIONSHIPS = [
    "family",
    "romantic",
    "friend_ally",
    "service_retainer",
    "enemy_rival",
    "no_relation",
]

NO_RELATION_LABEL = "no_relation"
POSITIVE_RELATIONS = [label for label in RELATIONSHIPS if label != NO_RELATION_LABEL]

RANDOM_SEED = 42
TEST_SIZE = 0.15
DEV_SIZE = 0.15
MAX_NEGATIVE_RATIO = 2.0
MIN_CONTEXT_CHARS = 25
EDGE_CONFIDENCE_THRESHOLD = 0.95  # predictions_all_best_baseline.csv

# -----------------------------------------------------------------------------
# Optional LLM Judge labeling configuration.
# -----------------------------------------------------------------------------
# Set this to True when you want Gemma to relabel data/candidate_examples.csv.
# Leave it False for cheap weak-label runs.
USE_LLM_JUDGE = False

# Gemma instruction-tuned model used as a classification-style judge.
# This model may require accepting the Gemma license on Hugging Face and being
# logged in with `huggingface-cli login` or by setting an HF token in the runtime.
LLM_JUDGE_MODEL_ID = "google/gemma-2-2b-it"

# The LLM Judge always reads the shared candidate file and writes a separate cache.
LLM_JUDGE_INPUT_CSV = CANDIDATE_EXAMPLES_CSV
LLM_JUDGE_OUTPUT_CSV = DATA_DIR / "candidate_examples_llm_labeled.csv"

# If False, an existing LLM-labeled CSV is reused and Gemma is not called again.
# Set to True only when you intentionally want to regenerate the expensive labels.
LLM_JUDGE_OVERWRITE_CACHE = False

# Inference settings. `device_map="auto"` is recommended on GPU runtimes.
# Use None if your local transformers/accelerate setup does not support device_map.
LLM_JUDGE_DEVICE_MAP = "auto"
LLM_JUDGE_TORCH_DTYPE = "auto"  # "auto", "bfloat16", "float16", or "float32"
LLM_JUDGE_MAX_NEW_TOKENS = 12
LLM_JUDGE_MAX_INPUT_CHARS = 1800
LLM_JUDGE_SAVE_EVERY = 25

print(f"XML path: {XML_PATH.resolve()}")
print(f"Shared data folder: {DATA_DIR.resolve()}")
print(f"Baseline output folder: {BASELINE_DIR.resolve()}")
print(f"BiLSTM output folder: {BILSTM_DIR.resolve()}")
print(f"Relationship labels: {RELATIONSHIPS}")
print(f"LLM Judge enabled: {USE_LLM_JUDGE}")
print(f"LLM Judge model: {LLM_JUDGE_MODEL_ID}")
print(f"LLM Judge input: {LLM_JUDGE_INPUT_CSV}")
print(f"LLM Judge cache: {LLM_JUDGE_OUTPUT_CSV}")


## 2. Imports

In [ ]:
import html
import json
import math
import re
import warnings
import xml.etree.ElementTree as ET
from collections import Counter, defaultdict
from pathlib import Path

import joblib
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.multiclass import OneVsRestClassifier
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_colwidth", 140)
np.random.seed(RANDOM_SEED)

## 3. Parse the MediaWiki XML

MediaWiki exports store the page title and page text as XML elements. This cell extracts them as:

- `character_name` = page `<title>`
- `character_text` = revision `<text>`

Characters pages that have less than 2000 characters are removed

In [ ]:
def get_xml_namespace(tag: str) -> str:
    """Return namespace prefix in ElementTree format, e.g. '{...}', or empty string."""
    if tag.startswith("{"):
        return tag.split("}", 1)[0] + "}"
    return ""


def parse_mediawiki_xml(xml_path: Path) -> pd.DataFrame:
    """Parse a MediaWiki XML export into one row per page."""
    xml_path = Path(xml_path)
    if not xml_path.exists():
        raise FileNotFoundError(f"XML file not found: {xml_path}")

    pages = []
    context = ET.iterparse(str(xml_path), events=("start", "end"))
    _, root = next(context)
    ns = get_xml_namespace(root.tag)

    for event, elem in context:
        if event == "end" and elem.tag == f"{ns}page":
            title = (elem.findtext(f"{ns}title") or "").strip()
            ns_id = (elem.findtext(f"{ns}ns") or "").strip()
            page_id = (elem.findtext(f"{ns}id") or "").strip()
            revision = elem.find(f"{ns}revision")
            text = ""
            timestamp = ""
            if revision is not None:
                text = revision.findtext(f"{ns}text") or ""
                timestamp = revision.findtext(f"{ns}timestamp") or ""

            is_redirect = elem.find(f"{ns}redirect") is not None or text.lstrip().upper().startswith("#REDIRECT")

            pages.append(
                {
                    "page_id": page_id,
                    "namespace": ns_id,
                    "character_name": title,
                    "character_text": text,
                    "revision_timestamp": timestamp,
                    "is_redirect": is_redirect,
                    "text_length": len(text),
                }
            )
            elem.clear()
            root.clear()

    return pd.DataFrame(pages)


pages_df = parse_mediawiki_xml(XML_PATH)

pages_df = pages_df[
    (pages_df["namespace"] == "0")
    & (~pages_df["is_redirect"])
    & (pages_df["text_length"] >= 2000)
].copy()

pages_df = pages_df.sort_values("character_name").reset_index(drop=True)
pages_df.to_csv(PAGES_CSV, index=False)

print(f"Parsed character pages with text_length >= 2000: {len(pages_df):,}")
pages_df[["page_id", "character_name", "text_length", "revision_timestamp"]].head(10)

## 4. Wikitext cleaning and character gazetteer

The character gazetteer is built from page titles. The cleaning functions remove common wiki markup while preserving readable text for sentence-level candidate generation.

In [ ]:
def normalize_title(title: str) -> str:
    """Normalize a MediaWiki title for matching."""
    title = html.unescape(str(title or ""))
    title = title.replace("_", " ").strip()
    title = re.sub(r"\s+", " ", title)
    return title


def strip_anchor(title: str) -> str:
    """Remove a #section anchor from a wiki title."""
    return normalize_title(str(title).split("#", 1)[0])


def extract_wikilinks(wikitext: str):
    """Extract wiki links as (target, display_text) pairs."""
    if not isinstance(wikitext, str):
        return []
    links = []
    for match in re.finditer(r"\[\[([^\]|#]+)(?:#[^\]|]*)?(?:\|([^\]]+))?\]\]", wikitext):
        target = strip_anchor(match.group(1))
        display_text = normalize_title(match.group(2) if match.group(2) else target)
        if target:
            links.append((target, display_text))
    return links


def remove_templates(text: str) -> str:
    """Remove balanced-looking {{...}} templates iteratively.

    This is a lightweight regex cleaner for a baseline notebook. It is not a full
    MediaWiki parser, but it is enough for candidate extraction and TF-IDF text.
    """
    pattern = re.compile(r"\{\{[^{}]*\}\}", flags=re.DOTALL)
    previous = None
    while previous != text:
        previous = text
        text = pattern.sub(" ", text)
    return text


def clean_wikitext(wikitext: str) -> str:
    """Convert raw wikitext into plain-ish text suitable for TF-IDF."""
    if not isinstance(wikitext, str):
        return ""

    text = html.unescape(wikitext)
    text = re.sub(r"<!--.*?-->", " ", text, flags=re.DOTALL)
    text = re.sub(r"<ref\b[^>/]*/>", " ", text, flags=re.IGNORECASE)
    text = re.sub(r"<ref\b[^>]*>.*?</ref>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"<gallery\b[^>]*>.*?</gallery>", " ", text, flags=re.IGNORECASE | re.DOTALL)
    text = re.sub(r"\[\[(?:Category|File|Image):[^\]]+\]\]", " ", text, flags=re.IGNORECASE)

    # Convert headings like {{h1|Story}} or {{h2|[[Part 4 Volume 3]]}} to text before template removal.
    text = re.sub(r"\{\{h[12]\|([^}|]+).*?\}\}", lambda m: f"\n{clean_wikitext(m.group(1))}\n", text, flags=re.IGNORECASE | re.DOTALL)

    # Convert wiki links to display text.
    text = re.sub(
        r"\[\[([^\]|#]+)(?:#[^\]|]*)?(?:\|([^\]]+))?\]\]",
        lambda m: normalize_title(m.group(2) if m.group(2) else m.group(1)),
        text,
    )

    text = re.sub(r"\[https?://[^\s\]]+\s+([^\]]+)\]", r"\1", text)
    text = re.sub(r"\[https?://[^\]]+\]", " ", text)
    text = re.sub(r"<br\s*/?>", "\n", text, flags=re.IGNORECASE)
    text = re.sub(r"<[^>]+>", " ", text)
    text = remove_templates(text)
    text = text.replace(chr(39) * 3, "").replace(chr(39) * 2, "")
    text = re.sub(r"={2,}[^=]+={2,}", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


characters = sorted({normalize_title(name) for name in pages_df["character_name"].dropna() if normalize_title(name)})
character_set = set(characters)

# Fast matcher for page-title character names. This avoids scanning every
# character name with a separate regex for every sentence.
def build_character_pattern(names: list[str]) -> re.Pattern:
    names = [name for name in names if len(name) >= 3]
    names = sorted(names, key=len, reverse=True)
    if not names:
        return re.compile(r"a^")  # matches nothing
    return re.compile(r"(?<![A-Za-z])(" + "|".join(re.escape(name) for name in names) + r")(?![A-Za-z])", flags=re.IGNORECASE)

CHARACTER_PATTERN = build_character_pattern(characters)
CHARACTER_LOOKUP = {name.casefold(): name for name in characters}

# Store a simple gazetteer for later inspection.
gazetteer_df = pd.DataFrame({"character_name": characters})
gazetteer_df.to_csv(CHARACTER_GAZETTEER_CSV, index=False)

print(f"Characters in gazetteer: {len(gazetteer_df):,}")
display(gazetteer_df.head(10))

## 5. Extract weak relationship signals from infoboxes

The MediaWiki character template often contains family fields such as `family/Father`, `family/Sister`, or `family/Spouse`. These provide useful weak labels.

This baseline maps:

- `family/Spouse`, `wife`, `husband`, `betrothed`, etc. → `romantic`
- other `family/...` fields → `family`

Only linked character names that also appear in the exported page-title gazetteer are kept.

In [ ]:
def extract_balanced_template(wikitext: str, template_name: str = "Character") -> str:
    """Return the first balanced {{Character ...}} template block, if found."""
    if not isinstance(wikitext, str):
        return ""
    match = re.search(r"\{\{\s*" + re.escape(template_name) + r"\b", wikitext, flags=re.IGNORECASE)
    if not match:
        return ""

    start = match.start()
    i = start
    depth = 0
    while i < len(wikitext) - 1:
        two = wikitext[i : i + 2]
        if two == "{{":
            depth += 1
            i += 2
            continue
        if two == "}}":
            depth -= 1
            i += 2
            if depth == 0:
                return wikitext[start:i]
            continue
        i += 1
    return ""


def parse_template_fields(template_text: str) -> dict:
    """Parse simple |key=value lines from a template.
    """
    fields = {}
    current_key = None
    for raw_line in template_text.splitlines():
        line = raw_line.strip()
        if line.startswith("|") and "=" in line:
            key, value = line[1:].split("=", 1)
            current_key = key.strip()
            fields[current_key] = value.strip()
        elif current_key and line and not line.startswith("}}"):  # continuation line
            fields[current_key] += " " + line
    return fields


def relation_from_infobox_field(key: str, value: str) -> str | None:
    """Map an infobox field key/value pair to one of the relationship labels."""
    key_l = key.lower().strip()
    value_l = value.lower().strip()

    romantic_markers = [
        "spouse",
        "wife",
        "husband",
        "lover",
        "fiance",
        "fiancée",
        "betrothed",
    ]

    service_retainer_markers = [
        "retainer",
        "retainers",
        "attendant",
        "attendants",
        "guard knight",
        "guard knights",
        "scholar",
        "scholars",
        "serves",
        "served",
        "serving",
    ]

    if any(marker in key_l for marker in romantic_markers):
        return "romantic" if "romantic" in RELATIONSHIPS else None

    if key_l.startswith("family/") or key_l in {"familytree", "relatives"}:
        return "family" if "family" in RELATIONSHIPS else None

    if any(marker in key_l for marker in service_retainer_markers):
        return "service_retainer" if "service_retainer" in RELATIONSHIPS else None

    if key_l == "occupation" and any(marker in value_l for marker in service_retainer_markers):
        return "service_retainer" if "service_retainer" in RELATIONSHIPS else None

    return None


def extract_infobox_relation_edges(row: pd.Series) -> list[dict]:
    """Extract weakly labeled relation examples from one page's Character infobox."""
    head = row["character_name"]
    template = extract_balanced_template(row["character_text"], "Character")
    fields = parse_template_fields(template)
    examples = []

    for key, value in fields.items():
        label = relation_from_infobox_field(key, value)
        if label is None:
            continue

        for target, display_text in extract_wikilinks(value):
            if target in character_set and target != head:
                evidence = clean_wikitext(value)
                examples.append(
                    {
                        "head": head,
                        "tail": target,
                        "context": f"Infobox field {key}: {evidence}",
                        "section": "infobox",
                        "source_type": "infobox",
                        "weak_label": label,
                        "weak_label_source": f"infobox:{key}",
                    }
                )
    return examples


infobox_examples = []
for _, row in pages_df.iterrows():
    infobox_examples.extend(extract_infobox_relation_edges(row))

infobox_df = pd.DataFrame(infobox_examples)
print(f"Infobox weak relation examples: {len(infobox_df):,}")
if len(infobox_df):
    display(infobox_df.head(10))
    display(infobox_df["weak_label"].value_counts())

## 6. Generate sentence-level candidate pairs

For each character page, this cell:

1. cleans the page text,
2. splits it into sentence-like contexts,
3. finds mentions of other exported character-page titles,
4. creates a candidate pair `(page character, mentioned character)`, and
5. assigns a weak label using relationship keywords.

These labels are noisy and should be treated as baseline/distant-supervision labels.

In [ ]:
RELATION_PATTERNS = {
    "family": [
        r"\bfather\b", r"\bmother\b", r"\bparent\b", r"\bparents\b", r"\bson\b", r"\bdaughter\b",
        r"\bsibling\b", r"\bbrother\b", r"\bsister\b", r"\buncle\b", r"\baunt\b", r"\bcousin\b",
        r"\bgrandfather\b", r"\bgrandmother\b", r"\brelative\b", r"\bin-law\b", r"\bniece\b", r"\bnephew\b",
    ],
    "romantic": [
        r"\bwife\b", r"\bhusband\b", r"\bspouse\b", r"\bmarried\b", r"\bmarriage\b",
        r"\bengaged\b", r"\bengagement\b", r"\bbetrothed\b", r"\bfianc[eé]\b",
        r"\blover\b", r"\blove interest\b", r"\bin love\b", r"\bromantic\b",
    ],
    "friend_ally": [
        r"\bfriend\b", r"\bfriends\b", r"\bfriendship\b",
        r"\bally\b", r"\ballies\b", r"\ballied\b", r"\bcompanion\b",
        r"\bclose friend\b", r"\bbest friend\b", r"\btrusted friend\b", r"\bconfidant\b",
    ],
    "service_retainer": [
        r"\bretainer\b", r"\bretainers\b", r"\bguard knight\b", r"\bguard knights\b",
        r"\battendant\b", r"\battendants\b",
        r"\bscholar\b", r"\bscholars\b",
        r"\bserved\b", r"\bserves\b", r"\bserving\b",
        r"\bin .* service\b", r"\bhead attendant\b", r"\bhead scholar\b", r"\bhead guard knight\b",
    ],
    "enemy_rival": [
        r"\benemy\b", r"\benemies\b", r"\brival\b", r"\brivals\b", r"\bopponent\b", r"\bopposes\b",
        r"\bbetray\b", r"\bbetrayed\b", r"\bkill(?:ed|s)?\b", r"\bexecute(?:d|s)?\b", r"\bhate(?:d|s)?\b",
        r"\babuse(?:d|s)?\b", r"\bjealous\b", r"\bpersecution\b", r"\battack(?:ed|s)?\b",
    ],
}

# Keep only patterns for labels present in RELATIONSHIPS.
RELATION_PATTERNS = {label: pats for label, pats in RELATION_PATTERNS.items() if label in RELATIONSHIPS}
RELATION_PRIORITY = ["romantic", "family", "enemy_rival", "service_retainer", "friend_ally"]
RELATION_PRIORITY = [label for label in RELATION_PRIORITY if label in RELATIONSHIPS]


def split_wikitext_into_sections(wikitext: str) -> list[tuple[str, str]]:
    """Split raw wikitext by Fandom-style {{h1|...}} / {{h2|...}} headings."""
    if not isinstance(wikitext, str):
        return [("lead", "")]

    heading_re = re.compile(r"\{\{h[12]\|([^}|]+)(?:\|[^}]*)?\}\}", flags=re.IGNORECASE | re.DOTALL)
    sections = []
    current_section = "lead"
    last = 0

    for match in heading_re.finditer(wikitext):
        if match.start() > last:
            sections.append((current_section, wikitext[last: match.start()]))
        current_section = clean_wikitext(match.group(1)) or "section"
        last = match.end()

    sections.append((current_section, wikitext[last:]))
    return sections


def split_sentences(text: str) -> list[str]:
    """Simple sentence splitter for cleaned wiki text."""
    text = re.sub(r"\s+", " ", text).strip()
    if not text:
        return []
    pieces = re.split(r"(?<=[.!?])\s+(?=[A-Z0-9'\"“])", text)
    return [piece.strip() for piece in pieces if len(piece.strip()) >= MIN_CONTEXT_CHARS]


def name_in_text(name: str, text: str) -> bool:
    """Case-insensitive character-name match with letter boundaries."""
    pattern = r"(?<![A-Za-z])" + re.escape(name) + r"(?![A-Za-z])"
    return re.search(pattern, text, flags=re.IGNORECASE) is not None


def find_mentioned_characters(text: str, head: str) -> list[str]:
    """Find other character names from the gazetteer mentioned in text."""
    found = []
    for match in CHARACTER_PATTERN.finditer(text):
        canonical = CHARACTER_LOOKUP.get(match.group(0).casefold())
        if canonical and canonical != head:
            found.append(canonical)
    return sorted(set(found))


def infer_weak_label_from_text(text: str, section: str = "") -> tuple[str, str]:
    """Infer a weak label from keyword patterns."""
    full_text = f"{section} {text}".lower()
    scores = {}
    matches = {}

    for label, patterns in RELATION_PATTERNS.items():
        label_matches = [pat for pat in patterns if re.search(pat, full_text, flags=re.IGNORECASE)]
        if label_matches:
            scores[label] = len(label_matches)
            matches[label] = label_matches

    if not scores:
        return NO_RELATION_LABEL, "keyword:none"

    # Highest score wins. Priority order breaks ties.
    max_score = max(scores.values())
    best_labels = [label for label, score in scores.items() if score == max_score]
    for label in RELATION_PRIORITY:
        if label in best_labels:
            return label, f"keyword:{','.join(matches[label][:3])}"
    return best_labels[0], f"keyword:{','.join(matches[best_labels[0]][:3])}"


def mark_entities(context: str, head: str, tail: str) -> str:
    """Add explicit entity markers for the target pair."""
    marked = context

    tail_pattern = re.compile(r"(?<![A-Za-z])" + re.escape(tail) + r"(?![A-Za-z])", flags=re.IGNORECASE)
    head_pattern = re.compile(r"(?<![A-Za-z])" + re.escape(head) + r"(?![A-Za-z])", flags=re.IGNORECASE)

    marked = tail_pattern.sub(lambda m: f"[TAIL] {m.group(0)} [/TAIL]", marked, count=1)

    if head_pattern.search(marked):
        marked = head_pattern.sub(lambda m: f"[HEAD] {m.group(0)} [/HEAD]", marked, count=1)
    else:
        marked = f"[HEAD] {head} [/HEAD] {marked}"

    return marked


def make_model_text(row: pd.Series, use_markers: bool = True, include_metadata: bool = True) -> str:
    """Build the text input for TF-IDF."""
    context = row["context"]
    if use_markers:
        context = mark_entities(context, row["head"], row["tail"])
    if include_metadata:
        return f"section={row['section']} source={row['source_type']} {context}"
    return context


def generate_sentence_candidates(pages: pd.DataFrame) -> pd.DataFrame:
    examples = []

    for _, row in pages.iterrows():
        head = row["character_name"]
        for section, raw_section in split_wikitext_into_sections(row["character_text"]):
            cleaned_section = clean_wikitext(raw_section)
            for sentence in split_sentences(cleaned_section):
                mentioned = find_mentioned_characters(sentence, head)
                for tail in mentioned:
                    weak_label, weak_source = infer_weak_label_from_text(sentence, section)
                    examples.append(
                        {
                            "head": head,
                            "tail": tail,
                            "context": sentence,
                            "section": section,
                            "source_type": "prose",
                            "weak_label": weak_label,
                            "weak_label_source": weak_source,
                        }
                    )

    return pd.DataFrame(examples)


sentence_df = generate_sentence_candidates(pages_df)
print(f"Sentence-level candidate examples: {len(sentence_df):,}")
if len(sentence_df):
    display(sentence_df.head(10))
    display(sentence_df["weak_label"].value_counts())

## 7. Build the initial candidate dataset

This cell creates `data/candidate_examples.csv`. The file contains every candidate pair used by both models, along with an initial weak `label` copied from `weak_label`.

If the optional LLM Judge is enabled in Section 8, the notebook will read this shared candidate file, generate `data/candidate_examples_llm_labeled.csv`, and replace `candidates_df` with the LLM-labeled version before training.


In [ ]:
candidate_parts = []
if len(infobox_df):
    candidate_parts.append(infobox_df)
if len(sentence_df):
    candidate_parts.append(sentence_df)

if not candidate_parts:
    raise ValueError("No candidate examples were generated. Check the XML path and character-page filter.")

candidates_df = pd.concat(candidate_parts, ignore_index=True)
candidates_df = candidates_df.drop_duplicates(subset=["head", "tail", "context", "source_type"]).reset_index(drop=True)

# Keep only labels included in the configured relationship list.
candidates_df = candidates_df[candidates_df["weak_label"].isin(RELATIONSHIPS)].copy()

# Balance negatives against positives so the model does not learn to predict only no_relation.
pos_df = candidates_df[candidates_df["weak_label"] != NO_RELATION_LABEL].copy()
neg_df = candidates_df[candidates_df["weak_label"] == NO_RELATION_LABEL].copy()

if len(pos_df) == 0:
    raise ValueError("No positive relationship examples were generated. Add more relation keywords or enable an external labeling workflow.")

max_negatives = int(MAX_NEGATIVE_RATIO * len(pos_df))
if len(neg_df) > max_negatives:
    neg_df = neg_df.sample(n=max_negatives, random_state=RANDOM_SEED)

candidates_df = pd.concat([pos_df, neg_df], ignore_index=True)
candidates_df = candidates_df.sample(frac=1.0, random_state=RANDOM_SEED).reset_index(drop=True)

candidates_df["label"] = candidates_df["weak_label"]
candidates_df["pair_id"] = candidates_df["head"] + " || " + candidates_df["tail"]
candidates_df["candidate_id"] = [f"cand_{i:06d}" for i in range(len(candidates_df))]
candidates_df["text_basic"] = candidates_df["context"]
candidates_df["text_marked"] = candidates_df.apply(lambda row: make_model_text(row, use_markers=True, include_metadata=True), axis=1)

candidates_df.to_csv(CANDIDATE_EXAMPLES_CSV, index=False)

label_counts = candidates_df["label"].value_counts().rename_axis("label").reset_index(name="count")
label_counts.to_csv(WEAK_LABEL_DISTRIBUTION_CSV, index=False)

print(f"Training candidates after balancing: {len(candidates_df):,}")
display(label_counts)
display(candidates_df[["candidate_id", "head", "tail", "label", "source_type", "context"]].head(10))

## 8. Optional LLM Judge labeling pipeline

This optional step adds a cached LLM Judge workflow for replacing rule-based weak labels before model training.

When `USE_LLM_JUDGE = True`, the notebook:

1. Reads every row from `data/candidate_examples.csv`.
2. Prompts a Gemma instruction-tuned model through Hugging Face `transformers`.
3. Forces the judge to choose exactly one label from `RELATIONSHIPS`.
4. Stores the judged label in the `label` column of `data/candidate_examples_llm_labeled.csv`.
5. Reuses that CSV in future runs instead of calling the LLM again.
6. Replaces `candidates_df` with the LLM-labeled file before the train/dev/test split.

This step is disabled by default because it is computationally expensive. Keep `USE_LLM_JUDGE = False` for cheap weak-label runs. Set `USE_LLM_JUDGE = True` only when you want to create or reuse the cached LLM-labeled training data.


In [ ]:
def validate_candidate_training_frame(df: pd.DataFrame, frame_name: str = "candidates_df") -> pd.DataFrame:
    """Validate the candidate DataFrame used by the downstream training pipeline."""
    required_cols = {
        "candidate_id",
        "head",
        "tail",
        "context",
        "source_type",
        "weak_label",
        "label",
        "pair_id",
        "text_basic",
        "text_marked",
    }
    missing_cols = sorted(required_cols - set(df.columns))
    if missing_cols:
        raise ValueError(f"{frame_name} is missing required columns: {missing_cols}")

    invalid_labels = sorted(set(df["label"].dropna().astype(str)) - set(RELATIONSHIPS))
    if invalid_labels:
        raise ValueError(f"{frame_name} contains labels not in RELATIONSHIPS: {invalid_labels}")

    if df["label"].isna().any():
        raise ValueError(f"{frame_name} contains missing labels.")

    return df.reset_index(drop=True).copy()


RELATION_LABEL_DESCRIPTIONS = {
    "family": "family or relative relationship, including parent, child, sibling, spouse's family, adoptive family, or blood relation",
    "romantic": "romantic relationship, marriage, lover, fiance, spouse, betrothal, or romantic interest",
    "friend_ally": "friendship, ally, trusted companion, teammate, or cooperative relationship",
    "service_retainer": "service, retainer, attendant, guard knight, scholar, subordinate, master-servant, or duty-based relationship",
    "enemy_rival": "enemy, rival, antagonist, betrayal, hostility, violence, hatred, or opposition",
    "no_relation": "the context does not explicitly state one of the listed relationships between head and tail",
}


def build_llm_judge_prompt(row: pd.Series) -> str:
    """Build one deterministic classification prompt for a candidate pair."""
    labels_block = "\n".join(
        f"- {label}: {RELATION_LABEL_DESCRIPTIONS.get(label, label)}"
        for label in RELATIONSHIPS
    )

    context = str(row.get("context", ""))
    if len(context) > LLM_JUDGE_MAX_INPUT_CHARS:
        context = context[:LLM_JUDGE_MAX_INPUT_CHARS].rstrip() + " ..."

    section = str(row.get("section", ""))
    source_type = str(row.get("source_type", ""))
    weak_label_source = str(row.get("weak_label_source", ""))

    return f"""
You are an NLP annotation judge for a character relationship extraction dataset.

Choose exactly one relationship label from this list:
{labels_block}

Decision rules:
- Use only the supplied context, page/section metadata, head character, and tail character.
- Label the relationship from HEAD to TAIL when the context explicitly supports it.
- If the context mentions both characters but does not clearly express one listed relationship, choose no_relation.
- If more than one relationship is possible, choose the most explicit relationship in the context.
- Return only the label string. Do not explain your decision.

HEAD: {row.get("head", "")}
TAIL: {row.get("tail", "")}
SECTION: {section}
SOURCE_TYPE: {source_type}
WEAK_LABEL_SOURCE: {weak_label_source}
CONTEXT:
{context}

LABEL:
""".strip()


def parse_llm_judge_label(raw_response: str, valid_labels: list[str]) -> str | None:
    """Parse the judge response into one allowed label, or return None if parsing fails."""
    if raw_response is None:
        return None

    text = str(raw_response).strip()
    text = re.sub(r"```(?:json|text)?", "", text, flags=re.IGNORECASE).replace("```", "").strip()

    # Exact normalized match first.
    normalized = text.strip().strip('"\'`.,:;()[]{}').casefold()
    label_lookup = {label.casefold(): label for label in valid_labels}
    if normalized in label_lookup:
        return label_lookup[normalized]

    # JSON-ish or "label: X" response.
    label_pattern = r"(?:label|relationship)\s*[\"']?\s*[:=]\s*[\"']?([A-Za-z_]+)"
    match = re.search(label_pattern, text, flags=re.IGNORECASE)
    if match:
        candidate = match.group(1).casefold()
        if candidate in label_lookup:
            return label_lookup[candidate]

    # Last-resort boundary match. Sort by length so no_relation is not partially shadowed.
    for label in sorted(valid_labels, key=len, reverse=True):
        if re.search(rf"(?<![A-Za-z_]){re.escape(label)}(?![A-Za-z_])", text, flags=re.IGNORECASE):
            return label

    return None


def resolve_llm_torch_dtype(dtype_name: str):
    """Convert the config string into a torch dtype understood by transformers."""
    import torch

    if dtype_name is None or str(dtype_name).lower() == "none":
        return None
    dtype_name = str(dtype_name).lower()
    if dtype_name == "auto":
        return "auto"

    dtype_map = {
        "bfloat16": torch.bfloat16,
        "bf16": torch.bfloat16,
        "float16": torch.float16,
        "fp16": torch.float16,
        "float32": torch.float32,
        "fp32": torch.float32,
    }
    if dtype_name not in dtype_map:
        raise ValueError(f"Unsupported LLM_JUDGE_TORCH_DTYPE: {dtype_name}")
    return dtype_map[dtype_name]


def load_llm_judge_model(model_id: str):
    """Load the Gemma judge lazily so disabled weak-label runs do not need transformers."""
    from transformers import AutoModelForCausalLM, AutoTokenizer

    model_kwargs = {}
    dtype = resolve_llm_torch_dtype(LLM_JUDGE_TORCH_DTYPE)
    if dtype is not None:
        model_kwargs["torch_dtype"] = dtype
    if LLM_JUDGE_DEVICE_MAP is not None:
        model_kwargs["device_map"] = LLM_JUDGE_DEVICE_MAP

    tokenizer = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(model_id, **model_kwargs)
    model.eval()

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    return tokenizer, model


def generate_llm_judge_response(tokenizer, model, prompt: str) -> str:
    """Generate one short deterministic label response from the LLM judge."""
    import torch

    messages = [{"role": "user", "content": prompt}]
    model_inputs = tokenizer.apply_chat_template(
        messages,
        tokenize=True,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
    )

    model_device = next(model.parameters()).device
    model_inputs = {key: value.to(model_device) for key, value in model_inputs.items()}

    with torch.inference_mode():
        output_ids = model.generate(
            **model_inputs,
            max_new_tokens=LLM_JUDGE_MAX_NEW_TOKENS,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id,
        )

    prompt_length = model_inputs["input_ids"].shape[-1]
    generated_ids = output_ids[0][prompt_length:]
    return tokenizer.decode(generated_ids, skip_special_tokens=True).strip()


def label_candidate_examples_with_llm(
    input_csv: Path,
    output_csv: Path,
    model_id: str,
) -> pd.DataFrame:
    """Read the configured candidate CSV, label every row with the LLM Judge, and save a new CSV."""
    input_csv = Path(input_csv)
    output_csv = Path(output_csv)
    if not input_csv.exists():
        raise FileNotFoundError(f"LLM Judge input CSV not found: {input_csv}")

    df = pd.read_csv(input_csv)
    df = validate_candidate_training_frame(df, frame_name="LLM Judge input")

    tokenizer, model = load_llm_judge_model(model_id)

    labels = []
    raw_responses = []
    parse_statuses = []
    partial_csv = output_csv.with_suffix(output_csv.suffix + ".partial")

    print(f"Running LLM Judge over {len(df):,} rows from {input_csv}")
    print(f"Model: {model_id}")

    for idx, row in df.iterrows():
        prompt = build_llm_judge_prompt(row)
        raw_response = generate_llm_judge_response(tokenizer, model, prompt)
        parsed_label = parse_llm_judge_label(raw_response, RELATIONSHIPS)

        if parsed_label is None:
            parsed_label = NO_RELATION_LABEL
            parse_status = "invalid_response_fallback_to_no_relation"
        else:
            parse_status = "parsed"

        labels.append(parsed_label)
        raw_responses.append(raw_response)
        parse_statuses.append(parse_status)

        if (idx + 1) % LLM_JUDGE_SAVE_EVERY == 0:
            partial_df = df.iloc[: idx + 1].copy()
            partial_df["label"] = labels
            partial_df["llm_judge_raw_response"] = raw_responses
            partial_df["llm_judge_parse_status"] = parse_statuses
            partial_df.to_csv(partial_csv, index=False)
            print(f"LLM Judge progress: {idx + 1:,}/{len(df):,} rows")

    df["label"] = labels
    df["llm_judge_raw_response"] = raw_responses
    df["llm_judge_parse_status"] = parse_statuses
    df["llm_judge_model"] = model_id

    output_csv.parent.mkdir(parents=True, exist_ok=True)
    df.to_csv(output_csv, index=False)

    if partial_csv.exists():
        partial_csv.unlink()

    return validate_candidate_training_frame(df, frame_name="LLM-labeled candidates")


def load_or_create_llm_labeled_candidates(
    input_csv: Path,
    output_csv: Path,
    model_id: str,
    overwrite_cache: bool = False,
) -> pd.DataFrame:
    """Load cached LLM labels if present; otherwise run the expensive judge once."""
    input_csv = Path(input_csv)
    output_csv = Path(output_csv)

    if output_csv.exists() and not overwrite_cache:
        print(f"LLM-labeled CSV already exists. Skipping LLM Judge and loading: {output_csv}")
        cached_df = pd.read_csv(output_csv)
        return validate_candidate_training_frame(cached_df, frame_name="cached LLM-labeled candidates")

    if output_csv.exists() and overwrite_cache:
        print(f"Regenerating LLM-labeled CSV because LLM_JUDGE_OVERWRITE_CACHE=True: {output_csv}")
    else:
        print(f"No LLM-labeled CSV found. Creating: {output_csv}")

    return label_candidate_examples_with_llm(input_csv=input_csv, output_csv=output_csv, model_id=model_id)


if USE_LLM_JUDGE:
    candidates_df = load_or_create_llm_labeled_candidates(
        input_csv=LLM_JUDGE_INPUT_CSV,
        output_csv=LLM_JUDGE_OUTPUT_CSV,
        model_id=LLM_JUDGE_MODEL_ID,
        overwrite_cache=LLM_JUDGE_OVERWRITE_CACHE,
    )
    LABEL_SOURCE = "llm_judge"
    print("Training pipeline will use LLM Judge labels.")
else:
    candidates_df = pd.read_csv(LLM_JUDGE_INPUT_CSV)
    candidates_df = validate_candidate_training_frame(candidates_df, frame_name="weak-labeled candidates")
    LABEL_SOURCE = "weak_labels"
    print(f"LLM Judge disabled. Training pipeline will use weak labels from {LLM_JUDGE_INPUT_CSV}.")

training_label_counts = candidates_df["label"].value_counts().rename_axis("label").reset_index(name="count")
training_label_counts.to_csv(TRAINING_LABEL_DISTRIBUTION_CSV, index=False)

print(f"Label source for this run: {LABEL_SOURCE}")
display(training_label_counts)

## 9. Train/dev/test split by character pair

The split uses `pair_id = head || tail` as the group. This reduces leakage, because the same character pair should not appear in both train and test.

In [ ]:
def group_split_dataframe(df: pd.DataFrame, group_col: str = "pair_id"):
    """Split into train/dev/test using grouped splits.

    If the grouped split fails due to very small data, fall back to stratified random splits.
    """
    df = df.copy().reset_index(drop=True)
    groups = df[group_col]

    try:
        splitter1 = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_SEED)
        train_dev_idx, test_idx = next(splitter1.split(df, groups=groups))
        train_dev_df = df.iloc[train_dev_idx].copy()
        test_df = df.iloc[test_idx].copy()

        relative_dev_size = DEV_SIZE / (1.0 - TEST_SIZE)
        splitter2 = GroupShuffleSplit(n_splits=1, test_size=relative_dev_size, random_state=RANDOM_SEED)
        train_idx, dev_idx = next(splitter2.split(train_dev_df, groups=train_dev_df[group_col]))
        train_df = train_dev_df.iloc[train_idx].copy()
        dev_df = train_dev_df.iloc[dev_idx].copy()
        split_method = "grouped_by_pair"
    except Exception as exc:
        print(f"Grouped split failed ({exc}). Falling back to stratified random split.")
        train_dev_df, test_df = train_test_split(
            df,
            test_size=TEST_SIZE,
            random_state=RANDOM_SEED,
            stratify=df["label"] if df["label"].nunique() > 1 else None,
        )
        relative_dev_size = DEV_SIZE / (1.0 - TEST_SIZE)
        train_df, dev_df = train_test_split(
            train_dev_df,
            test_size=relative_dev_size,
            random_state=RANDOM_SEED,
            stratify=train_dev_df["label"] if train_dev_df["label"].nunique() > 1 else None,
        )
        split_method = "stratified_random"

    return (
        train_df.reset_index(drop=True),
        dev_df.reset_index(drop=True),
        test_df.reset_index(drop=True),
        split_method,
    )


train_df, dev_df, test_df, split_method = group_split_dataframe(candidates_df)

if train_df["label"].nunique() < 2:
    raise ValueError("Training split has fewer than two classes. Add more labels/examples or reduce filtering.")

train_df.to_csv(TRAIN_CSV, index=False)
dev_df.to_csv(DEV_CSV, index=False)
test_df.to_csv(TEST_CSV, index=False)

print(f"Split method: {split_method}")
print(f"Train: {len(train_df):,} | Dev: {len(dev_df):,} | Test: {len(test_df):,}")

split_summary = pd.concat(
    [
        train_df["label"].value_counts().rename("train"),
        dev_df["label"].value_counts().rename("dev"),
        test_df["label"].value_counts().rename("test"),
    ],
    axis=1,
).fillna(0).astype(int)
split_summary.to_csv(SPLIT_LABEL_DISTRIBUTION_CSV)
display(split_summary)

## 10. Train TF-IDF + Logistic Regression baseline variations

This notebook trains two variants of the same baseline model family:

1. `tfidf_logreg_basic`: TF-IDF over the raw context only.
2. `tfidf_logreg_marked`: TF-IDF over entity-marked context with section/source metadata.

The second version corresponds to the feature-engineering variation from the proposal.

In [ ]:
def build_tfidf_logreg_pipeline() -> Pipeline:
    return Pipeline(
        steps=[
            (
                "tfidf",
                TfidfVectorizer(
                    lowercase=True,
                    ngram_range=(1, 3),
                    min_df=1,
                    max_df=0.95,
                    max_features=50_000,
                    sublinear_tf=True,
                    strip_accents="unicode",
                ),
            ),
            (
                "clf",
                LogisticRegression(
                    max_iter=2000,
                    C=10.0,
                    class_weight=None,
                    solver="lbfgs",
                    tol=1e-4,
                    random_state=RANDOM_SEED,
                ),
            ),
        ]
    )

def evaluate_predictions(y_true, y_pred, labels_order: list[str]) -> dict:
    labels_present = [label for label in labels_order if label in set(y_true) | set(y_pred)]
    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "macro_f1": f1_score(y_true, y_pred, labels=labels_present, average="macro", zero_division=0),
        "weighted_f1": f1_score(y_true, y_pred, labels=labels_present, average="weighted", zero_division=0),
    }


def train_and_evaluate_variant(variant_name: str, text_column: str):
    print(f"\nTraining variant: {variant_name}")
    model = build_tfidf_logreg_pipeline()
    model.fit(train_df[text_column], train_df["label"])

    dev_pred = model.predict(dev_df[text_column])
    test_pred = model.predict(test_df[text_column])

    dev_metrics = evaluate_predictions(dev_df["label"], dev_pred, RELATIONSHIPS)
    test_metrics = evaluate_predictions(test_df["label"], test_pred, RELATIONSHIPS)

    metrics_row = {
        "variant": variant_name,
        "text_column": text_column,
        "dev_accuracy": dev_metrics["accuracy"],
        "dev_macro_f1": dev_metrics["macro_f1"],
        "dev_weighted_f1": dev_metrics["weighted_f1"],
        "test_accuracy": test_metrics["accuracy"],
        "test_macro_f1": test_metrics["macro_f1"],
        "test_weighted_f1": test_metrics["weighted_f1"],
        "train_examples": len(train_df),
        "dev_examples": len(dev_df),
        "test_examples": len(test_df),
    }

    # Save model.
    model_path = BASELINE_DIR / f"{variant_name}.joblib"
    joblib.dump(model, model_path)

    # Save classification report and confusion matrix for the test set.
    labels_present = [label for label in RELATIONSHIPS if label in set(test_df["label"]) | set(test_pred)]
    report = classification_report(
        test_df["label"],
        test_pred,
        labels=labels_present,
        output_dict=True,
        zero_division=0,
    )
    report_df = pd.DataFrame(report).transpose()
    report_df.to_csv(BASELINE_DIR / f"classification_report_{variant_name}.csv")

    cm = confusion_matrix(test_df["label"], test_pred, labels=labels_present)
    cm_df = pd.DataFrame(cm, index=[f"true_{x}" for x in labels_present], columns=[f"pred_{x}" for x in labels_present])
    cm_df.to_csv(BASELINE_DIR / f"confusion_matrix_{variant_name}.csv")

    # Save test predictions.
    test_out = test_df.copy()
    test_out["predicted_label"] = test_pred
    if hasattr(model.named_steps["clf"], "predict_proba"):
        probs = model.predict_proba(test_df[text_column])
        classes = model.named_steps["clf"].classes_
        test_out["confidence"] = probs.max(axis=1)
        for i, label in enumerate(classes):
            test_out[f"prob_{label}"] = probs[:, i]
    test_out.to_csv(BASELINE_DIR / f"predictions_test_{variant_name}.csv", index=False)

    return model, metrics_row


variants = {
    "tfidf_logreg_basic": "text_basic",
    "tfidf_logreg_marked": "text_marked",
}

trained_models = {}
metrics_rows = []
for variant_name, text_column in variants.items():
    model, metrics = train_and_evaluate_variant(variant_name, text_column)
    trained_models[variant_name] = {"model": model, "text_column": text_column}
    metrics_rows.append(metrics)

metrics_df = pd.DataFrame(metrics_rows).sort_values("dev_macro_f1", ascending=False).reset_index(drop=True)
metrics_df.to_csv(BASELINE_DIR / "metrics_baseline_variants.csv", index=False)

display(metrics_df)

## 11. Train BiLSTM relation classifier

This section implements the second model: a basic neural relation classifier using a bidirectional LSTM.

The model reuses the same `train_df`, `dev_df`, and `test_df` splits as the TF-IDF baseline. It uses the `text_marked` column, which includes explicit `[HEAD]` and `[TAIL]` markers so the model knows which two characters the label refers to.

Architecture:

```text
entity-marked text
  -> regex tokenizer
  -> vocabulary IDs
  -> embedding layer
  -> bidirectional LSTM
  -> final forward/backward hidden states
  -> dropout
  -> linear classification layer
  -> softmax relationship label
```


In [ ]:
# BiLSTM relation classifier.
# This cell intentionally uses only the existing train/dev/test DataFrames so the comparison
# with the baseline is made on exactly the same split.

import copy
import random
from collections import Counter

try:
    import torch
    import torch.nn as nn
    from torch.nn.utils.rnn import pack_padded_sequence
    from torch.utils.data import DataLoader, Dataset
except ImportError as exc:
    raise ImportError(
        "PyTorch is required for the BiLSTM section. Install torch in the notebook environment and rerun this cell."
    ) from exc

# BILSTM_DIR is configured in Section 1 so model-specific outputs stay separate from shared data files.
BILSTM_DIR.mkdir(parents=True, exist_ok=True)

BILSTM_TEXT_COLUMN = "text_marked"
BILSTM_MAX_VOCAB_SIZE = 20_000
BILSTM_MIN_TOKEN_FREQ = 1
BILSTM_MAX_LEN = 128
BILSTM_EMBEDDING_DIM = 100
BILSTM_HIDDEN_DIM = 128
BILSTM_NUM_LAYERS = 1
BILSTM_DROPOUT = 0.30
BILSTM_BATCH_SIZE = min(32, max(2, len(train_df)))
BILSTM_EPOCHS = 8
BILSTM_LEARNING_RATE = 1e-3
BILSTM_WEIGHT_DECAY = 1e-5
BILSTM_PATIENCE = 2

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"
PAD_IDX = 0
UNK_IDX = 1

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Limiting CPU threads avoids occasional OpenMP/MKL slowdowns or deadlocks on small notebook workloads.
if DEVICE.type == "cpu":
    torch.set_num_threads(1)
print(f"BiLSTM device: {DEVICE}")


def set_reproducible_seed(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_reproducible_seed(RANDOM_SEED)

# Keep entity markers as their own tokens. Lowercasing makes the vocabulary smaller.
TOKEN_PATTERN = re.compile(r"\[/?(?:HEAD|TAIL)\]|[A-Za-z]+(?:'[A-Za-z]+)?|\d+|[^\w\s]", flags=re.IGNORECASE)


def tokenize_for_bilstm(text: str) -> list[str]:
    return [token.lower() for token in TOKEN_PATTERN.findall(str(text or ""))]


def build_bilstm_vocab(texts: pd.Series, max_vocab_size: int = BILSTM_MAX_VOCAB_SIZE, min_freq: int = BILSTM_MIN_TOKEN_FREQ) -> dict[str, int]:
    counts = Counter()
    for text in texts.fillna(""):
        counts.update(tokenize_for_bilstm(text))

    vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    for token, count in counts.most_common(max_vocab_size - len(vocab)):
        if count >= min_freq and token not in vocab:
            vocab[token] = len(vocab)
    return vocab


vocab = build_bilstm_vocab(train_df[BILSTM_TEXT_COLUMN])
label_to_id = {label: idx for idx, label in enumerate(RELATIONSHIPS)}
id_to_label = {idx: label for label, idx in label_to_id.items()}

print(f"BiLSTM vocabulary size: {len(vocab):,}")
print(f"Number of labels: {len(label_to_id)}")


class RelationDataset(Dataset):
    def __init__(self, df: pd.DataFrame, text_column: str, vocab: dict[str, int], label_to_id: dict[str, int], max_len: int):
        self.df = df.reset_index(drop=True).copy()
        self.text_column = text_column
        self.vocab = vocab
        self.label_to_id = label_to_id
        self.max_len = max_len

    def __len__(self) -> int:
        return len(self.df)

    def encode_text(self, text: str) -> list[int]:
        tokens = tokenize_for_bilstm(text)[: self.max_len]
        if not tokens:
            tokens = [UNK_TOKEN]
        return [self.vocab.get(token, UNK_IDX) for token in tokens]

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]
        input_ids = self.encode_text(row[self.text_column])
        label_id = self.label_to_id[row["label"]]
        return {
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "length": len(input_ids),
            "label": torch.tensor(label_id, dtype=torch.long),
        }


def collate_relation_batch(batch: list[dict]) -> dict[str, torch.Tensor]:
    lengths = torch.tensor([item["length"] for item in batch], dtype=torch.long)
    labels = torch.stack([item["label"] for item in batch])
    max_len = int(lengths.max().item())

    input_ids = torch.full((len(batch), max_len), PAD_IDX, dtype=torch.long)
    for row_idx, item in enumerate(batch):
        ids = item["input_ids"]
        input_ids[row_idx, : len(ids)] = ids

    return {"input_ids": input_ids, "lengths": lengths, "labels": labels}


def make_bilstm_loader(df: pd.DataFrame, shuffle: bool = False) -> DataLoader:
    dataset = RelationDataset(df, BILSTM_TEXT_COLUMN, vocab, label_to_id, BILSTM_MAX_LEN)
    return DataLoader(
        dataset,
        batch_size=BILSTM_BATCH_SIZE,
        shuffle=shuffle,
        collate_fn=collate_relation_batch,
    )


class BiLSTMRelationClassifier(nn.Module):
    def __init__(
        self,
        vocab_size: int,
        num_labels: int,
        embedding_dim: int = BILSTM_EMBEDDING_DIM,
        hidden_dim: int = BILSTM_HIDDEN_DIM,
        num_layers: int = BILSTM_NUM_LAYERS,
        dropout: float = BILSTM_DROPOUT,
        pad_idx: int = PAD_IDX,
    ):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embedding_dim, padding_idx=pad_idx)
        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if num_layers > 1 else 0.0,
        )
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden_dim * 2, num_labels)

    def forward(self, input_ids: torch.Tensor, lengths: torch.Tensor) -> torch.Tensor:
        embedded = self.embedding(input_ids)
        packed = pack_padded_sequence(
            embedded,
            lengths.detach().cpu(),
            batch_first=True,
            enforce_sorted=False,
        )
        _, (hidden, _) = self.lstm(packed)

        # hidden shape: [num_layers * 2, batch, hidden_dim]
        # Last layer forward state is hidden[-2], backward state is hidden[-1].
        representation = torch.cat([hidden[-2], hidden[-1]], dim=1)
        representation = self.dropout(representation)
        return self.classifier(representation)


def predict_bilstm(model: nn.Module, df: pd.DataFrame) -> tuple[np.ndarray, np.ndarray]:
    model.eval()
    loader = make_bilstm_loader(df, shuffle=False)
    all_probs = []

    with torch.no_grad():
        for batch in loader:
            input_ids = batch["input_ids"].to(DEVICE)
            lengths = batch["lengths"].to(DEVICE)
            logits = model(input_ids, lengths)
            probs = torch.softmax(logits, dim=1).detach().cpu().numpy()
            all_probs.append(probs)

    probs = np.vstack(all_probs) if all_probs else np.empty((0, len(RELATIONSHIPS)))
    pred_ids = probs.argmax(axis=1) if len(probs) else np.array([], dtype=int)
    pred_labels = np.array([id_to_label[int(idx)] for idx in pred_ids])
    return pred_labels, probs


train_loader = make_bilstm_loader(train_df, shuffle=True)

y_counts = train_df["label"].value_counts()
class_weights = []
for label in RELATIONSHIPS:
    # Use inverse-frequency weighting so minority relation classes matter during training.
    count = max(1, int(y_counts.get(label, 0)))
    class_weights.append(len(train_df) / (len(RELATIONSHIPS) * count))
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(DEVICE)

bilstm_model = BiLSTMRelationClassifier(
    vocab_size=len(vocab),
    num_labels=len(RELATIONSHIPS),
).to(DEVICE)

criterion = nn.CrossEntropyLoss(weight=class_weights)
optimizer = torch.optim.AdamW(
    bilstm_model.parameters(),
    lr=BILSTM_LEARNING_RATE,
    weight_decay=BILSTM_WEIGHT_DECAY,
)

best_dev_macro_f1 = -1.0
best_state = None
epochs_without_improvement = 0
bilstm_history_rows = []

for epoch in range(1, BILSTM_EPOCHS + 1):
    bilstm_model.train()
    total_loss = 0.0
    total_examples = 0

    for batch in train_loader:
        input_ids = batch["input_ids"].to(DEVICE)
        lengths = batch["lengths"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        optimizer.zero_grad()
        logits = bilstm_model(input_ids, lengths)
        loss = criterion(logits, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(bilstm_model.parameters(), max_norm=1.0)
        optimizer.step()

        batch_size = labels.size(0)
        total_loss += float(loss.item()) * batch_size
        total_examples += batch_size

    train_loss = total_loss / max(1, total_examples)
    dev_pred, _ = predict_bilstm(bilstm_model, dev_df)
    dev_metrics = evaluate_predictions(dev_df["label"], dev_pred, RELATIONSHIPS)

    bilstm_history_rows.append(
        {
            "epoch": epoch,
            "train_loss": train_loss,
            "dev_accuracy": dev_metrics["accuracy"],
            "dev_macro_f1": dev_metrics["macro_f1"],
            "dev_weighted_f1": dev_metrics["weighted_f1"],
        }
    )

    print(
        f"Epoch {epoch:02d} | loss={train_loss:.4f} | "
        f"dev_macro_f1={dev_metrics['macro_f1']:.4f} | dev_accuracy={dev_metrics['accuracy']:.4f}"
    )

    if dev_metrics["macro_f1"] > best_dev_macro_f1:
        best_dev_macro_f1 = dev_metrics["macro_f1"]
        best_state = copy.deepcopy(bilstm_model.state_dict())
        epochs_without_improvement = 0
    else:
        epochs_without_improvement += 1
        if epochs_without_improvement >= BILSTM_PATIENCE:
            print(f"Early stopping after {epoch} epochs.")
            break

if best_state is not None:
    bilstm_model.load_state_dict(best_state)

bilstm_history_df = pd.DataFrame(bilstm_history_rows)
bilstm_history_df.to_csv(BILSTM_DIR / "training_history.csv", index=False)

dev_pred, dev_probs = predict_bilstm(bilstm_model, dev_df)
test_pred, test_probs = predict_bilstm(bilstm_model, test_df)

dev_metrics = evaluate_predictions(dev_df["label"], dev_pred, RELATIONSHIPS)
test_metrics = evaluate_predictions(test_df["label"], test_pred, RELATIONSHIPS)

bilstm_metrics = {
    "variant": "bilstm_marked",
    "text_column": BILSTM_TEXT_COLUMN,
    "dev_accuracy": dev_metrics["accuracy"],
    "dev_macro_f1": dev_metrics["macro_f1"],
    "dev_weighted_f1": dev_metrics["weighted_f1"],
    "test_accuracy": test_metrics["accuracy"],
    "test_macro_f1": test_metrics["macro_f1"],
    "test_weighted_f1": test_metrics["weighted_f1"],
    "train_examples": len(train_df),
    "dev_examples": len(dev_df),
    "test_examples": len(test_df),
    "vocab_size": len(vocab),
    "max_len": BILSTM_MAX_LEN,
    "embedding_dim": BILSTM_EMBEDDING_DIM,
    "hidden_dim": BILSTM_HIDDEN_DIM,
    "epochs_run": int(len(bilstm_history_df)),
    "best_dev_macro_f1": float(best_dev_macro_f1),
}

bilstm_metrics_df = pd.DataFrame([bilstm_metrics])
bilstm_metrics_df.to_csv(BILSTM_DIR / "metrics_bilstm.csv", index=False)

labels_present = [label for label in RELATIONSHIPS if label in set(test_df["label"]) | set(test_pred)]
bilstm_report = classification_report(
    test_df["label"],
    test_pred,
    labels=labels_present,
    output_dict=True,
    zero_division=0,
)
pd.DataFrame(bilstm_report).transpose().to_csv(BILSTM_DIR / "classification_report_bilstm.csv")

bilstm_cm = confusion_matrix(test_df["label"], test_pred, labels=labels_present)
bilstm_cm_df = pd.DataFrame(
    bilstm_cm,
    index=[f"true_{label}" for label in labels_present],
    columns=[f"pred_{label}" for label in labels_present],
)
bilstm_cm_df.to_csv(BILSTM_DIR / "confusion_matrix_bilstm.csv")

bilstm_test_out = test_df.copy()
bilstm_test_out["predicted_label"] = test_pred
bilstm_test_out["confidence"] = test_probs.max(axis=1) if len(test_probs) else []
for label_idx, label in enumerate(RELATIONSHIPS):
    bilstm_test_out[f"prob_{label}"] = test_probs[:, label_idx] if len(test_probs) else []
bilstm_test_out.to_csv(BILSTM_DIR / "predictions_test_bilstm.csv", index=False)

# Save a checkpoint and the vocabulary so the model can be reused later.
torch.save(
    {
        "model_state_dict": bilstm_model.state_dict(),
        "vocab": vocab,
        "label_to_id": label_to_id,
        "config": {
            "max_len": BILSTM_MAX_LEN,
            "embedding_dim": BILSTM_EMBEDDING_DIM,
            "hidden_dim": BILSTM_HIDDEN_DIM,
            "num_layers": BILSTM_NUM_LAYERS,
            "dropout": BILSTM_DROPOUT,
        },
    },
    BILSTM_DIR / "bilstm_relation_classifier.pt",
)

# Predict all candidates so the same KG aggregation code can be reused later.
bilstm_all_pred = candidates_df.copy()
bilstm_all_labels, bilstm_all_probs = predict_bilstm(bilstm_model, bilstm_all_pred)
bilstm_all_pred["predicted_label"] = bilstm_all_labels
bilstm_all_pred["confidence"] = bilstm_all_probs.max(axis=1) if len(bilstm_all_probs) else []
for label_idx, label in enumerate(RELATIONSHIPS):
    bilstm_all_pred[f"prob_{label}"] = bilstm_all_probs[:, label_idx] if len(bilstm_all_probs) else []

# Keep structured infobox relations as high-confidence weak labels only during weak-label runs.
# When LLM Judge labels are enabled, avoid reintroducing weak labels into KG generation.
if LABEL_SOURCE == "weak_labels":
    structured_mask = (
        (bilstm_all_pred["source_type"] == "infobox")
        & (bilstm_all_pred["weak_label"] != NO_RELATION_LABEL)
    )
    bilstm_all_pred.loc[structured_mask, "predicted_label"] = bilstm_all_pred.loc[structured_mask, "weak_label"]
    bilstm_all_pred.loc[structured_mask, "confidence"] = 0.95
else:
    print("LLM Judge labels are enabled; skipping infobox weak-label override for BiLSTM KG predictions.")

bilstm_all_predictions_path = BILSTM_DIR / "predictions_all_bilstm.csv"
bilstm_all_pred.to_csv(bilstm_all_predictions_path, index=False)

print(f"Saved BiLSTM outputs to: {BILSTM_DIR}")
display(bilstm_metrics_df)
display(bilstm_history_df.tail())
display(bilstm_test_out[["head", "tail", "label", "predicted_label", "confidence", "context"]].head(10))


## 12. Choose the best baseline variant and predict all candidates

The best baseline variant is selected by dev macro-F1. Predictions for all candidate pairs are saved for later KG construction.

In [ ]:
best_variant = metrics_df.iloc[0]["variant"]
best_text_column = metrics_df.iloc[0]["text_column"]
best_model = trained_models[best_variant]["model"]

print(f"Best variant by dev macro-F1: {best_variant} using {best_text_column}")

all_pred = candidates_df.copy()
all_pred["predicted_label"] = best_model.predict(all_pred[best_text_column])

if hasattr(best_model.named_steps["clf"], "predict_proba"):
    probs = best_model.predict_proba(all_pred[best_text_column])
    classes = best_model.named_steps["clf"].classes_
    all_pred["confidence"] = probs.max(axis=1)
    for i, label in enumerate(classes):
        all_pred[f"prob_{label}"] = probs[:, i]
else:
    all_pred["confidence"] = np.nan

if LABEL_SOURCE == "weak_labels":
    structured_mask = (
        (all_pred["source_type"] == "infobox")
        & (all_pred["weak_label"] != NO_RELATION_LABEL)
    )

    all_pred.loc[structured_mask, "predicted_label"] = all_pred.loc[structured_mask, "weak_label"]
    all_pred.loc[structured_mask, "confidence"] = 0.95
else:
    print("LLM Judge labels are enabled; skipping infobox weak-label override for baseline KG predictions.")

all_predictions_path = BASELINE_DIR / "predictions_all_best_baseline.csv"
all_pred.to_csv(all_predictions_path, index=False)
print(f"Saved all predictions to: {all_predictions_path}")
display(all_pred[["head", "tail", "label", "predicted_label", "confidence", "context"]].head(10))

## 13. Aggregate predictions into Knowledge Graph edges

This cell converts mention-level predictions into graph edges:

```text
head_character -- predicted_relationship --> tail_character
```

It removes `no_relation`, applies a confidence threshold, groups repeated evidence for the same pair/relation, and keeps the strongest relation for each pair.

In [ ]:
# Relations such as family, romantic, friendship, and rivalry are treated as undirected:
# A --family--> B and B --family--> A are merged into one canonical edge.
# Service/retainer relations are treated as directed because direction matters:
# A --service_retainer--> B is not equivalent to B --service_retainer--> A.
UNDIRECTED_RELATIONS = {
    "family",
    "romantic",
    "friend_ally",
    "enemy_rival",
} & set(RELATIONSHIPS)

DIRECTED_RELATIONS = {
    "service_retainer",
} & set(RELATIONSHIPS)

RELATIONS_WITH_DIRECTION_RULES = UNDIRECTED_RELATIONS | DIRECTED_RELATIONS
UNSPECIFIED_RELATIONS = set(POSITIVE_RELATIONS) - RELATIONS_WITH_DIRECTION_RULES

if UNSPECIFIED_RELATIONS:
    print(
        "Warning: These positive relations are not listed in UNDIRECTED_RELATIONS or DIRECTED_RELATIONS "
        "and will be treated as directed:",
        sorted(UNSPECIFIED_RELATIONS),
    )

print(f"Undirected relations: {sorted(UNDIRECTED_RELATIONS)}")
print(f"Directed relations: {sorted(DIRECTED_RELATIONS)}")


def canonicalize_edge(row: pd.Series) -> pd.Series:
    """Create canonical edge keys.

    For undirected relations, sorted(head, tail) is used so mirrored predictions
    collapse into one edge. For directed relations, the original head/tail order is kept.
    """
    head = row["head"]
    tail = row["tail"]
    relation = row["predicted_label"]

    if relation in UNDIRECTED_RELATIONS:
        edge_head, edge_tail = sorted([head, tail])
        edge_direction = "undirected"
    else:
        # Directed relations and any unspecified positive relation keep model direction.
        edge_head, edge_tail = head, tail
        edge_direction = "directed"

    return pd.Series(
        {
            "edge_head": edge_head,
            "edge_tail": edge_tail,
            "edge_direction": edge_direction,
        }
    )


def aggregate_kg_edges(predictions: pd.DataFrame, confidence_threshold: float = EDGE_CONFIDENCE_THRESHOLD) -> pd.DataFrame:
    """Aggregate mention-level predictions into KG edges.

    Mirrored edges are merged for labels in UNDIRECTED_RELATIONS.
    Directed labels keep their original head -> tail direction.
    """
    edges = predictions.copy()
    edges = edges[edges["predicted_label"] != NO_RELATION_LABEL].copy()
    edges = edges[edges["confidence"].fillna(1.0) >= confidence_threshold].copy()

    empty_columns = [
        "head",
        "tail",
        "relation",
        "edge_direction",
        "evidence_count",
        "mean_confidence",
        "max_confidence",
        "edge_score",
        "evidence",
    ]

    if len(edges) == 0:
        return pd.DataFrame(columns=empty_columns)

    edge_keys = edges.apply(canonicalize_edge, axis=1)
    edges = pd.concat([edges, edge_keys], axis=1)

    grouped_rows = []
    group_cols = ["edge_head", "edge_tail", "predicted_label", "edge_direction"]

    for (edge_head, edge_tail, relation, edge_direction), group in edges.groupby(group_cols):
        evidence_count = len(group)
        mean_conf = float(group["confidence"].mean())
        max_conf = float(group["confidence"].max())
        edge_score = mean_conf * math.log1p(evidence_count)

        evidence = " | ".join(
            group
            .sort_values("confidence", ascending=False)["context"]
            .dropna()
            .astype(str)
            .drop_duplicates()
            .head(3)
            .tolist()
        )

        grouped_rows.append(
            {
                "head": edge_head,
                "tail": edge_tail,
                "relation": relation,
                "edge_direction": edge_direction,
                "evidence_count": evidence_count,
                "mean_confidence": mean_conf,
                "max_confidence": max_conf,
                "edge_score": edge_score,
                "evidence": evidence,
            }
        )

    kg_edges = pd.DataFrame(grouped_rows)

    # Keep only the highest-scoring relation per canonical character pair.
    # For undirected relations, this removes mirrored duplicates such as A-B and B-A.
    # For directed relations, the original direction is preserved.
    kg_edges = (
        kg_edges.sort_values(["head", "tail", "edge_score"], ascending=[True, True, False])
        .drop_duplicates(subset=["head", "tail"], keep="first")
        .sort_values("edge_score", ascending=False)
        .reset_index(drop=True)
    )

    return kg_edges


kg_edges_df = aggregate_kg_edges(all_pred)
kg_edges_path = BASELINE_DIR / "kg_edges_best_baseline.csv"
kg_edges_df.to_csv(kg_edges_path, index=False)

print(f"KG edges saved to: {kg_edges_path}")
print(f"Predicted nodes: {len(set(kg_edges_df['head']).union(set(kg_edges_df['tail'])) if len(kg_edges_df) else set())}")
print(f"Predicted edges: {len(kg_edges_df):,}")

if len(kg_edges_df):
    print("\nEdge direction counts:")
    display(kg_edges_df["edge_direction"].value_counts().rename_axis("edge_direction").reset_index(name="count"))

display(kg_edges_df.head(20))


## 14. Aggregate BiLSTM predictions into Knowledge Graph edges

This cell applies the same KG aggregation function to the BiLSTM predictions so that the baseline and neural model are compared using the same graph-construction logic.


In [ ]:
if "bilstm_all_pred" in globals():
    kg_edges_bilstm_df = aggregate_kg_edges(bilstm_all_pred)
    kg_edges_bilstm_path = BILSTM_DIR / "kg_edges_bilstm.csv"
    kg_edges_bilstm_df.to_csv(kg_edges_bilstm_path, index=False)

    print(f"BiLSTM KG edges saved to: {kg_edges_bilstm_path}")
    print(
        "BiLSTM predicted nodes:",
        len(set(kg_edges_bilstm_df["head"]).union(set(kg_edges_bilstm_df["tail"]))) if len(kg_edges_bilstm_df) else 0,
    )
    print(f"BiLSTM predicted edges: {len(kg_edges_bilstm_df):,}")

    if len(kg_edges_bilstm_df):
        print("\nBiLSTM edge direction counts:")
        display(kg_edges_bilstm_df["edge_direction"].value_counts().rename_axis("edge_direction").reset_index(name="count"))

    display(kg_edges_bilstm_df.head(20))
else:
    print("BiLSTM predictions were not found. Run Section 11 before this cell.")


## 15. Create PyVis Knowledge Graph visualization

If `pyvis` is installed, this cell creates an interactive HTML graph. If not, it writes a simpler HTML edge table so the pipeline still completes.

In [ ]:
def inject_relation_filter(html_path: Path, relations: list[str]) -> None:
    """Inject relation checkbox filters into a PyVis HTML file."""
    html_path = Path(html_path)
    html_text = html_path.read_text(encoding="utf-8")

    checkbox_html = "\n".join(
        f"""
        <label style="display:block; margin: 3px 0;">
            <input type="checkbox" class="relation-filter" value="{relation}" checked>
            {relation}
        </label>
        """
        for relation in relations
    )

    control_panel = f"""
    <div id="relation-filter-panel" style="
        position: fixed;
        top: 10px;
        right: 10px;
        z-index: 9999;
        background: white;
        border: 1px solid #ccc;
        border-radius: 6px;
        padding: 10px 12px;
        font-family: Arial, sans-serif;
        font-size: 13px;
        box-shadow: 0 2px 8px rgba(0,0,0,0.15);
        max-width: 240px;
    ">
        <strong>Filter relations</strong>
        <div style="margin-top: 6px;">
            {checkbox_html}
        </div>
        <button id="select-all-relations" style="margin-top: 8px;">Select all</button>
        <button id="clear-all-relations" style="margin-top: 8px;">Clear all</button>
    </div>
    """

    filter_script = """
    <script type="text/javascript">
    document.addEventListener("DOMContentLoaded", function () {
        if (typeof edges === "undefined" || typeof nodes === "undefined" || typeof network === "undefined") {
            console.warn("PyVis variables not found; relation filter was not attached.");
            return;
        }

        var allEdges = edges.get();
        var allNodes = nodes.get();

        function getEdgeRelation(edge) {
            // Prefer custom relation metadata.
            // Fall back to edge label, because PyVis always keeps the label.
            return edge.relation || edge.label;
        }

        function updateGraphFilter() {
            var checkedRelations = Array.from(
                document.querySelectorAll(".relation-filter:checked")
            ).map(function (box) {
                return box.value;
            });

            var visibleNodeIds = new Set();
            var edgeUpdates = [];

            allEdges.forEach(function (edge) {
                var relation = getEdgeRelation(edge);
                var isVisible = checkedRelations.includes(relation);

                edgeUpdates.push({
                    id: edge.id,
                    hidden: !isVisible
                });

                if (isVisible) {
                    visibleNodeIds.add(edge.from);
                    visibleNodeIds.add(edge.to);
                }
            });

            var nodeUpdates = allNodes.map(function (node) {
                return {
                    id: node.id,
                    hidden: !visibleNodeIds.has(node.id)
                };
            });

            edges.update(edgeUpdates);
            nodes.update(nodeUpdates);

            network.redraw();
        }

        document.querySelectorAll(".relation-filter").forEach(function (box) {
            box.addEventListener("change", updateGraphFilter);
        });

        document.getElementById("select-all-relations").addEventListener("click", function () {
            document.querySelectorAll(".relation-filter").forEach(function (box) {
                box.checked = true;
            });
            updateGraphFilter();
        });

        document.getElementById("clear-all-relations").addEventListener("click", function () {
            document.querySelectorAll(".relation-filter").forEach(function (box) {
                box.checked = false;
            });
            updateGraphFilter();
        });
    });
    </script>
    """

    html_text = html_text.replace("<body>", f"<body>\n{control_panel}")
    html_text = html_text.replace("</body>", f"{filter_script}\n</body>")

    html_path.write_text(html_text, encoding="utf-8")

def write_pyvis_graph(edges_df: pd.DataFrame, output_path: Path, title: str = "TF-IDF + Logistic Regression KG"):
    """Write an interactive PyVis graph if pyvis is available; otherwise write a fallback HTML table."""
    output_path = Path(output_path)

    if len(edges_df) == 0:
        output_path.write_text("<html><body><h1>No edges passed the threshold.</h1></body></html>", encoding="utf-8")
        return "empty"

    try:
        from pyvis.network import Network

        net = Network(height="800px", width="100%", directed=True, notebook=False)
        net.barnes_hut()

        net.set_options("""
        {
          "physics": {
            "enabled": true,
            "barnesHut": {
              "gravitationalConstant": -25000,
              "centralGravity": 1,
              "springLength": 150,
              "springConstant": 0.04,
              "damping": 0.18,
              "avoidOverlap": 1
            },
            "stabilization": {
              "enabled": true,
              "iterations": 1200,
              "updateInterval": 25
            }
          },
          "nodes": {
            "font": {
              "size": 30
            }
          },
          "edges": {
            "font": {
              "size": 8
            },
            "smooth": {
              "enabled": true,
              "type": "dynamic"
            }
          },
          "interaction": {
            "dragNodes": true,
            "dragView": true,
            "zoomView": true
          }
        }
        """)

        degree_counter = Counter(edges_df["head"]) + Counter(edges_df["tail"])
        nodes = sorted(set(edges_df["head"]).union(set(edges_df["tail"])))

        for node in nodes:
            degree = degree_counter[node]
            net.add_node(
                node,
                label=node,
                size=min(35, 10 + 2 * degree),
                title=f"Character: {node}<br>Degree: {degree}",
            )

        for edge_idx, (_, row) in enumerate(edges_df.iterrows()):
            edge_direction = row.get("edge_direction", "directed")
            tooltip = (
                f"Relation: {row['relation']}<br>"
                f"Direction: {edge_direction}<br>"
                f"Mean confidence: {row['mean_confidence']:.3f}<br>"
                f"Evidence count: {int(row['evidence_count'])}<br>"
                f"Evidence: {row['evidence']}"
            )

            # Keep arrows only for directed relations such as service_retainer.
            # Undirected relations such as family, romantic, friend_ally, and enemy_rival
            # are displayed without arrowheads.
            if edge_direction == "undirected":
                arrows = {
                    "to": {"enabled": False},
                    "from": {"enabled": False},
                    "middle": {"enabled": False},
                }
            else:
                arrows = {
                    "to": {"enabled": True},
                    "from": {"enabled": False},
                    "middle": {"enabled": False},
                }

            net.add_edge(
                row["head"],
                row["tail"],
                label=row["relation"],
                width=0.5,
                arrows=arrows,
                title=tooltip,
                id=f"edge_{edge_idx}",
                relation=row["relation"],
                edge_direction=edge_direction,
            )

        net.write_html(str(output_path))
        relations = sorted(edges_df["relation"].dropna().unique().tolist())
        inject_relation_filter(output_path, relations)
        return "pyvis"

    except Exception as exc:
        fallback_html = f"""
        <html>
        <head><meta charset=\"utf-8\"><title>{title}</title></head>
        <body>
        <h1>{title}</h1>
        <p>PyVis graph could not be created: {html.escape(str(exc))}</p>
        {edges_df.to_html(index=False, escape=True)}
        </body>
        </html>
        """
        output_path.write_text(fallback_html, encoding="utf-8")
        return "fallback_html"


graph_path = BASELINE_DIR / "kg_tfidf_logreg_best_baseline.html"
graph_status = write_pyvis_graph(
    kg_edges_df,
    graph_path,
    title=f"Baseline KG: {best_variant}",
)

print(f"Graph status: {graph_status}")
print(f"Graph written to: {graph_path}")

if "kg_edges_bilstm_df" in globals():
    graph_path_bilstm = BILSTM_DIR / "kg_bilstm.html"
    graph_status_bilstm = write_pyvis_graph(
        kg_edges_bilstm_df,
        graph_path_bilstm,
        title="BiLSTM KG",
    )
    print(f"BiLSTM graph status: {graph_status_bilstm}")
    print(f"BiLSTM graph written to: {graph_path_bilstm}")
else:
    print("BiLSTM KG edges were not found. Run Section 14 before creating the BiLSTM graph.")


## 16. Output summary

This cell writes a compact JSON summary to `data/model_run_summary.json`. The summary records the shared data folder, model-specific output folders, LLM Judge settings, and the files produced by the run.


In [ ]:
summary = {
    "xml_path": str(XML_PATH),
    "data_dir": str(DATA_DIR),
    "baseline_output_dir": str(BASELINE_DIR),
    "bilstm_output_dir": str(BILSTM_DIR),
    "relationship_labels": RELATIONSHIPS,
    "num_pages": int(len(pages_df)),
    "num_candidates": int(len(candidates_df)),
    "num_train": int(len(train_df)),
    "num_dev": int(len(dev_df)),
    "num_test": int(len(test_df)),
    "split_method": split_method,
    "label_source": LABEL_SOURCE,
    "use_llm_judge": bool(USE_LLM_JUDGE),
    "llm_judge_model_id": str(LLM_JUDGE_MODEL_ID),
    "llm_judge_input_csv": str(LLM_JUDGE_INPUT_CSV),
    "llm_judge_output_csv": str(LLM_JUDGE_OUTPUT_CSV),
    "candidate_examples_csv": str(CANDIDATE_EXAMPLES_CSV),
    "train_csv": str(TRAIN_CSV),
    "dev_csv": str(DEV_CSV),
    "test_csv": str(TEST_CSV),
    "best_baseline_variant": str(best_variant),
    "best_baseline_text_column": str(best_text_column),
    "num_kg_edges_baseline": int(len(kg_edges_df)),
    "edge_confidence_threshold": float(EDGE_CONFIDENCE_THRESHOLD),
}

if "bilstm_metrics_df" in globals():
    summary["bilstm_variant"] = str(bilstm_metrics_df.iloc[0]["variant"])
    summary["bilstm_dev_macro_f1"] = float(bilstm_metrics_df.iloc[0]["dev_macro_f1"])
    summary["bilstm_test_macro_f1"] = float(bilstm_metrics_df.iloc[0]["test_macro_f1"])

if "kg_edges_bilstm_df" in globals():
    summary["num_kg_edges_bilstm"] = int(len(kg_edges_bilstm_df))

if "graph_status" in globals():
    summary["baseline_graph_status"] = graph_status
if "graph_status_bilstm" in globals():
    summary["bilstm_graph_status"] = graph_status_bilstm

summary["shared_data_files"] = sorted(str(path) for path in DATA_DIR.rglob("*") if path.is_file())
summary["baseline_files_written"] = sorted(str(path) for path in BASELINE_DIR.rglob("*") if path.is_file())
summary["bilstm_files_written"] = sorted(str(path) for path in BILSTM_DIR.rglob("*") if path.is_file())

summary_path = RUN_SUMMARY_JSON
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))


## 17. Compare baseline and BiLSTM results

This section combines the baseline and BiLSTM metrics into a single comparison table. It also compares the resulting Knowledge Graphs by number of nodes, number of edges, average confidence, and relation distribution.


In [ ]:
comparison_frames = []

if "metrics_df" in globals():
    baseline_comparison = metrics_df.copy()
    baseline_comparison["model_family"] = "TF-IDF + Logistic Regression"
    baseline_comparison["model_name"] = baseline_comparison["variant"]
    comparison_frames.append(baseline_comparison)

if "bilstm_metrics_df" in globals():
    bilstm_comparison = bilstm_metrics_df.copy()
    bilstm_comparison["model_family"] = "BiLSTM"
    bilstm_comparison["model_name"] = bilstm_comparison["variant"]
    comparison_frames.append(bilstm_comparison)

if comparison_frames:
    model_comparison_df = pd.concat(comparison_frames, ignore_index=True, sort=False)
    ordered_cols = [
        "model_family",
        "model_name",
        "text_column",
        "dev_accuracy",
        "dev_macro_f1",
        "dev_weighted_f1",
        "test_accuracy",
        "test_macro_f1",
        "test_weighted_f1",
        "train_examples",
        "dev_examples",
        "test_examples",
    ]
    ordered_cols = [col for col in ordered_cols if col in model_comparison_df.columns]
    model_comparison_df = model_comparison_df[ordered_cols].sort_values(
        ["dev_macro_f1", "test_macro_f1"], ascending=False
    ).reset_index(drop=True)
    model_comparison_df.to_csv(MODEL_COMPARISON_CSV, index=False)

    print("Mention-level model comparison:")
    display(model_comparison_df)

    best_by_dev = model_comparison_df.iloc[0]
    print(
        f"Best model by dev macro-F1: {best_by_dev['model_name']} "
        f"({best_by_dev['model_family']}) with dev_macro_f1={best_by_dev['dev_macro_f1']:.4f}."
    )
else:
    print("No metric DataFrames were found. Run the model-training sections first.")


def summarize_kg_edges(model_name: str, edges_df: pd.DataFrame) -> dict:
    if edges_df is None or len(edges_df) == 0:
        return {
            "model_name": model_name,
            "num_nodes": 0,
            "num_edges": 0,
            "mean_confidence": np.nan,
            "mean_evidence_count": np.nan,
            "relation_distribution": "",
        }

    nodes = set(edges_df["head"]).union(set(edges_df["tail"]))
    relation_counts = edges_df["relation"].value_counts().to_dict()
    relation_distribution = "; ".join(f"{label}: {count}" for label, count in relation_counts.items())

    return {
        "model_name": model_name,
        "num_nodes": int(len(nodes)),
        "num_edges": int(len(edges_df)),
        "mean_confidence": float(edges_df["mean_confidence"].mean()),
        "mean_evidence_count": float(edges_df["evidence_count"].mean()),
        "relation_distribution": relation_distribution,
    }

kg_summary_rows = []
if "kg_edges_df" in globals():
    kg_summary_rows.append(summarize_kg_edges(f"baseline_best: {best_variant}", kg_edges_df))
if "kg_edges_bilstm_df" in globals():
    kg_summary_rows.append(summarize_kg_edges("bilstm_marked", kg_edges_bilstm_df))

if kg_summary_rows:
    kg_model_comparison_df = pd.DataFrame(kg_summary_rows)
    kg_model_comparison_df.to_csv(KG_MODEL_COMPARISON_CSV, index=False)

    print("\nKnowledge Graph comparison:")
    display(kg_model_comparison_df)
else:
    print("No KG edge DataFrames were found. Run the KG aggregation sections first.")


## 18. What to report for the baseline and BiLSTM

In your project report, use the generated files to describe:

- the number of pages parsed from the XML,
- the number of candidate pairs,
- the label source used for the run: weak labels or LLM Judge labels,
- the shared training files under `data/`, especially `data/candidate_examples.csv`, `data/train.csv`, `data/dev.csv`, and `data/test.csv`,
- the label distribution from `data/training_label_distribution.csv`,
- the LLM Judge configuration if enabled: model ID, input CSV, output CSV, cache behavior, and prompt rules,
- the difference between the basic TF-IDF baseline and the entity-marker/section-feature TF-IDF variation,
- the BiLSTM architecture and training settings,
- macro-F1 and per-class F1 from the classification reports,
- the model comparison table from `data/metrics_model_comparison.csv`,
- the KG comparison table from `data/kg_model_comparison.csv`,
- the number of nodes and edges in the generated KG files,
- qualitative graph observations from `baseline/kg_tfidf_logreg_best_baseline.html` and `bilstm/kg_bilstm.html`.

Important limitation: weak labels are cheap but noisy. LLM Judge labels may be more semantically useful, but they are still model-generated labels rather than human-verified gold annotations. For final evaluation, use a manually verified dev/test subset if possible.
